Build a simplified Liquid Neural Network (LNN) in PyTorch to model continuous-time robot balance control under changing ground conditions. This end-to-end project demonstrates continuous-time neural dynamics, Euler integration, internal state evolution, synthetic sequence generation, model training, and simulation of a soccer-playing robot adapting its balance to varying surface slipperiness. :contentReference[oaicite:0]{index=0}

In [1]:
# ===========================================================================
# 0. Imports
# ===========================================================================

import torch
import torch.nn as nn
import torch.optim as optim

# ===========================================================================
# 1. Reproducibility and Device
# ===========================================================================

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ===========================================================================
# 2. Liquid Dynamics Model
# ===========================================================================

class LiquidDynamics(nn.Module):
    """
    Learns the continuous-time state equation:

        dz/dt = f(z, u)

    z:
        The robot's current internal balance-correction state.

    u:
        The current slipperiness sensor reading.

    The network receives [z, u] and predicts dz/dt, which tells us
    how quickly the internal state should change.
    """

    def __init__(self, hidden_size=16):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, z, u):
        """
        Args:
            z: Current state, shape (batch_size, 1)
            u: Current input, shape (batch_size, 1)

        Returns:
            dz_dt: Predicted rate of state change
        """

        state_and_input = torch.cat([z, u], dim=1)
        dz_dt = self.network(state_and_input)

        return dz_dt

# ===========================================================================
# 3. Continuous-Time State Update
# ===========================================================================

def evolve_state(model, z, u, interval=0.1, solver_steps=5):
    """
    Evolves the state over one observation interval using Euler integration.

    The interval is divided into several small solver steps. At every small
    step, the model predicts dz/dt and updates the state:

        z_new = z_old + dt * dz/dt

    Args:
        model:
            The LiquidDynamics network.

        z:
            Current internal state.

        u:
            Current slipperiness input.

        interval:
            Time between two sensor observations.

        solver_steps:
            Number of numerical integration steps inside the interval.

    Returns:
        Updated state after the complete interval.
    """

    dt = interval / solver_steps

    for _ in range(solver_steps):
        dz_dt = model(z, u)
        z = z + dt * dz_dt

    return z

# ===========================================================================
# 4. Simulate a Complete Input Sequence
# ===========================================================================

def simulate_sequence(model, initial_state, input_sequence):
    """
    Processes a sequence of slipperiness values.

    The final state from one interval becomes the initial state for
    the next interval. This gives the model memory across time.
    """

    z = initial_state
    predicted_states = []

    for u in input_sequence:
        u = u.view(1, 1)

        # Evolve the internal state continuously until the next reading.
        z = evolve_state(
            model=model,
            z=z,
            u=u,
            interval=0.1,
            solver_steps=5
        )

        predicted_states.append(z)

    return torch.cat(predicted_states, dim=0)

# ===========================================================================
# 5. Create Synthetic Training Sequences
# ===========================================================================

def create_training_batch(batch_size=32, sequence_length=20):
    """
    Creates synthetic slipperiness sequences.

    The desired balance correction is defined as:

        target correction = 0.1 + 0.8 * slipperiness

    This gives the network a simple relationship to learn:
    more slippery ground requires a larger correction.

    Real robot training data would come from sensors and measured
    control outcomes rather than this synthetic formula.
    """

    input_sequences = torch.rand(
        batch_size,
        sequence_length,
        1,
        device=device
    )

    target_sequences = 0.1 + 0.8 * input_sequences

    return input_sequences, target_sequences

# ===========================================================================
# 6. Train the Liquid Dynamics Model
# ===========================================================================

model = LiquidDynamics(hidden_size=16).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

num_epochs = 300
batch_size = 32
sequence_length = 20

print("\nTraining the continuous-time dynamics model...")

for epoch in range(num_epochs):
    model.train()

    input_batch, target_batch = create_training_batch(
        batch_size=batch_size,
        sequence_length=sequence_length
    )

    batch_loss = 0.0

    # Process each sequence independently for clarity.
    for batch_index in range(batch_size):
        input_sequence = input_batch[batch_index]
        target_sequence = target_batch[batch_index]

        initial_state = torch.zeros(
            1,
            1,
            device=device
        )

        predicted_sequence = simulate_sequence(
            model=model,
            initial_state=initial_state,
            input_sequence=input_sequence
        )

        batch_loss = batch_loss + criterion(
            predicted_sequence,
            target_sequence
        )

    batch_loss = batch_loss / batch_size

    optimizer.zero_grad()
    batch_loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Loss: {batch_loss.item():.6f}"
        )

print("Training complete.")

# ===========================================================================
# 7. Test the Model on Changing Ground Conditions
# ===========================================================================

model.eval()

# The robot begins on stable ground, encounters a slippery region,
# and then returns to a more stable surface.
slipperiness_sequence = torch.tensor(
    [
        [0.1],
        [0.1],
        [0.1],
        [0.2],
        [0.3],
        [0.8],
        [0.8],
        [0.8],
        [0.5],
        [0.2]
    ],
    dtype=torch.float32,
    device=device
)

initial_state = torch.tensor(
    [[0.0]],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    predicted_states = simulate_sequence(
        model=model,
        initial_state=initial_state,
        input_sequence=slipperiness_sequence
    )

# ===========================================================================
# 8. Display the Results
# ===========================================================================

print(
    "\nSimulation complete. "
    "Internal state evolution with changing slipperiness:"
)

for step in range(len(slipperiness_sequence)):
    time_value = (step + 1) * 0.1

    slipperiness = slipperiness_sequence[step].item()
    predicted_state = predicted_states[step].item()

    target_state = 0.1 + 0.8 * slipperiness

    if slipperiness < 0.3:
        condition = "stable"
    elif slipperiness < 0.7:
        condition = "changing"
    else:
        condition = "slippery"

    print(
        f"Step {step + 1:02d} | "
        f"t={time_value:.1f} | "
        f"surface={condition:8s} | "
        f"u={slipperiness:.1f} | "
        f"state(z)={predicted_state:.3f} | "
        f"target={target_state:.3f}"
    )


Using device: cpu

Training the continuous-time dynamics model...
Epoch [50/300], Loss: 0.058945
Epoch [100/300], Loss: 0.027029
Epoch [150/300], Loss: 0.012315
Epoch [200/300], Loss: 0.005390
Epoch [250/300], Loss: 0.002393
Epoch [300/300], Loss: 0.001465
Training complete.

Simulation complete. Internal state evolution with changing slipperiness:
Step 01 | t=0.1 | surface=stable   | u=0.1 | state(z)=0.129 | target=0.180
Step 02 | t=0.2 | surface=stable   | u=0.1 | state(z)=0.145 | target=0.180
Step 03 | t=0.3 | surface=stable   | u=0.1 | state(z)=0.146 | target=0.180
Step 04 | t=0.4 | surface=stable   | u=0.2 | state(z)=0.225 | target=0.260
Step 05 | t=0.5 | surface=changing | u=0.3 | state(z)=0.313 | target=0.340
Step 06 | t=0.6 | surface=slippery | u=0.8 | state(z)=0.708 | target=0.740
Step 07 | t=0.7 | surface=slippery | u=0.8 | state(z)=0.761 | target=0.740
Step 08 | t=0.8 | surface=slippery | u=0.8 | state(z)=0.768 | target=0.740
Step 09 | t=0.9 | surface=changing | u=0.5 | stat